In [1]:
import torch
from summarizer import Summarizer
from datasets import load_dataset

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_summ = Summarizer("distilbert-base-uncased", hidden_concat = True, hidden = [-1, -2], gpu_id = 0)

dataset = load_dataset("dennlinger/eur-lex-sum", "english")

In [10]:
import numpy as np

np.random.choice([1,2], size=3)

array([1, 2, 2])

In [ ]:
import warnings

warnings.filterwarnings("ignore")

def get_extractive_summary(doc, limit_sentences = 10):
    return model_summ(doc, use_first = False, return_as_list = True, num_sentences = limit_sentences) 

In [ ]:
import os
import json

path = "eurlex_ext"
if path not in os.listdir():
    os.mkdir(path)

iterator = zip(dataset["test"]["reference"], dataset["test"]["summary"])
for ref_id, (reference, orig_summary) in tqdm(enumerate(iterator)):
    if f"{ref_id}.json" in os.listdir(path):
        continue
    
    extractive_sum = get_extractive_summary(reference, 10)

    json.dump([extractive_sum], open(f"{path}/bert/{ref_id}.json", "w"), indent=2)


In [ ]:
from groq_functions_eurlex import llmResponse

llm_response = llmResponse("openai", "gpt-3.5", "basic", 188, 0)

In [ ]:
llm_response()

In [2]:
import evaluate

bertscore = evaluate.load("bertscore")
rouge = evaluate.load("rouge")

In [3]:
import os
import json
def load_predicted_data(path):
    predicted = []
    ordered_files = os.listdir(path)
    ordered_files = sorted(ordered_files, key = lambda x: int(x.split(".")[0]))
    for file in ordered_files:
        predicted.append(json.load(open(path+file, "r"))[-1]["content"])

    return predicted

ans = load_predicted_data("answers_ext/mixtral-8x7b-32768/basic/bert/")

In [4]:
rouge.compute(predictions=ans, references=dataset["test"]["summary"], use_stemmer = True)

{'rouge1': 0.30838205017163345,
 'rouge2': 0.1043526358031641,
 'rougeL': 0.1460263898384283,
 'rougeLsum': 0.28459712992987996}

In [6]:
bert_scores = bertscore.compute(predictions=ans, references=dataset["test"]["summary"], model_type="microsoft/deberta-large-mnli", batch_size = 3)

In [11]:
import numpy as np

np.mean(bert_scores["f1"])

0.5477348244253625